In [46]:
import pandas as pd
import glob

1-Importando a base de dados e juntando varias bases de dados da Leishmaniose
 

In [47]:
arquivos = glob.glob("LEIVA/*csv")

dfs = [
    pd.read_csv(arq, encoding="latin1", low_memory=False, sep=",")
    for arq in arquivos
]

df_total = pd.concat(dfs, ignore_index=True)


1.1-Mostrando a base de dados do SUS da doença Leishmaniose

In [48]:
df_total.head()

,ID_MUNICIP,ID_UNIDADE,DT_NOTIFIC,CS_RACA,CS_ESCOLAR,NU_ANO,SEM_NOT,SG_UF_NOT,ID_REGIONA,DT_SIN_PRI,...,DS_TRANS_1,DT_DESLC2,DS_MUN_2,CO_UF_2,CO_PAIS_2,DS_TRANS_2,DT_DESLC3,DS_MUN_3,CO_UF_3,CO_PAIS_3
0,2926301,2601567.0,20000809,9.0,9.0,2000.0,322000.0,BA,2.0,20000102,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2901502,2660229.0,20001122,4.0,3.0,2000.0,472000.0,BA,2.0,20001102,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2910800,2799758.0,20001215,4.0,9.0,2000.0,502000.0,BA,2.0,20001215,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2104552,2464659.0,20000810,4.0,6.0,2000.0,322000.0,MA,11.0,20000625,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2105302,2456540.0,20001016,NaN,6.0,2000.0,422000.0,MA,11.0,20000720,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


2.Esse código está filtrando colunas do DataFrame, mantendo apenas algumas com base em percentual de valores nulos e em uma lista de colunas obrigatórias.

In [49]:
manter = ['DT_OBITO', 'EVOLUCAO', 'CS_GESTANT']

df_limpo = df_total.loc[
    :, (df_total.isna().mean() <= 0.15) | (df_total.columns.isin(manter))
]


2.1.Mostrando o dataframe depois da limpeza

In [50]:
df_limpo.head()



,ID_MUNICIP,ID_UNIDADE,DT_NOTIFIC,CS_RACA,NU_ANO,SEM_NOT,SG_UF_NOT,DT_SIN_PRI,SEM_PRI,ANO_NASC,...,EMAGRA,TOSSE,BACO,FIGADO,HIV,IFI,OUTRO,CS_GESTANT,EVOLUCAO,DT_OBITO
0,2926301,2601567.0,20000809,9.0,2000.0,322000.0,BA,20000102,12000.0,1972.0,...,1.0,1.0,1.0,1.0,9.0,1.0,1.0,NaN,NaN,NaN
1,2901502,2660229.0,20001122,4.0,2000.0,472000.0,BA,20001102,442000.0,1984.0,...,2.0,2.0,1.0,1.0,2.0,1.0,1.0,NaN,NaN,NaN
2,2910800,2799758.0,20001215,4.0,2000.0,502000.0,BA,20001215,502000.0,1958.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,NaN,NaN,NaN
3,2104552,2464659.0,20000810,4.0,2000.0,322000.0,MA,20000625,262000.0,1998.0,...,1.0,2.0,1.0,1.0,2.0,3.0,3.0,NaN,NaN,NaN
4,2105302,2456540.0,20001016,NaN,2000.0,422000.0,MA,20000720,292000.0,1999.0,...,2.0,2.0,1.0,1.0,2.0,1.0,1.0,NaN,NaN,NaN


3.Excluindo colunas desnecessario para o projeto

In [51]:
df_limpo = df_limpo.drop(columns=['EMAGRA', 'TOSSE', 'BACO', 'FIGADO','IFI','OUTRO','ID_PAIS','FEBRE','FRAQUEZA','DT_SIN_PRI','SEM_PRI', 'CS_GESTANT','CS_RACA','HIV','EVOLUCAO','ID_MN_RESI','SEM_NOT'])


3.1.Visualizando base depois da exclusão

In [52]:
df_limpo.head()

,ID_MUNICIP,ID_UNIDADE,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ANO_NASC,CS_SEXO,SG_UF,DT_OBITO
0,2926301,2601567.0,20000809,2000.0,BA,1972.0,F,BA,NaN
1,2901502,2660229.0,20001122,2000.0,BA,1984.0,M,BA,NaN
2,2910800,2799758.0,20001215,2000.0,BA,1958.0,F,BA,NaN
3,2104552,2464659.0,20000810,2000.0,MA,1998.0,F,MA,NaN
4,2105302,2456540.0,20001016,2000.0,MA,1999.0,F,MA,NaN


3.2.Salvando nova base do SUS tratada.

In [53]:
df_limpo.to_csv("dados_sus_tratados.csv", index=False)

4-Descobrindo a linha onde começa o cabeçalho da base de dados?

In [54]:
with open('CODIGO_IBGE.csv', encoding='latin1') as f:
    for i, linha in enumerate(f):
        if linha.startswith('UF,'):
            print(i, linha)
            break



6 UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município,Distrito,Código de Distrito Completo,Nome_Distrito,OBS,,Código de Distrito Completo,Nome_Distrito,,,,,,,,,



4.1-Descobrindo q é o numero 6 onde começa o cabeçalho, fazendo a leitura e o tratamento dos dados

In [55]:
df_ibge = pd.read_csv(
    'CODIGO_IBGE.csv',
    sep=',',
    encoding='latin1',
    skiprows=6,
    header=0,
    engine='python'
)



4.2-Mostrado as linhas e as colunas vazias e não vazias

In [56]:
df_ibge.head()
df_ibge.columns


Index(['UF', 'Nome_UF', 'Região Geográfica Intermediária',
       'Nome Região Geográfica Intermediária', 'Região Geográfica Imediata',
       'Nome Região Geográfica Imediata', 'Município',
       'Código Município Completo', 'Nome_Município', 'Distrito',
       'Código de Distrito Completo', 'Nome_Distrito', 'OBS', 'Unnamed: 13',
       'Código de Distrito Completo.1', 'Nome_Distrito.1', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24'],
      dtype='object')

5-Removendo todas as colunas Unnamed.

In [57]:
df_ibge = df_ibge.loc[:, ~df_ibge.columns.str.startswith('Unnamed')]


6-Mostrando a base de dados do IBGE depois da limpeza

In [58]:
df_ibge.head()

,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município,Distrito,Código de Distrito Completo,Nome_Distrito,OBS,Código de Distrito Completo.1,Nome_Distrito.1
0,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,5,110001505,Alta Floresta D'Oeste,NaN,110001505.0,Alta Floresta D'Oeste
1,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,15,110001515,Filadélfia d'Oeste,NaN,110001515.0,Filadélfia d'Oeste
2,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,20,110001520,Izidolândia,NaN,110001520.0,Izidolândia
3,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,25,110001525,Nova Gease d'Oeste,NaN,110001525.0,Nova Gease d'Oeste
4,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,30,110001530,Rolim de Moura do Guaporé,NaN,110001530.0,Rolim de Moura do Guaporé


6.1.Excluindo colunas desncessarias para a analise

In [59]:
df_ibge = df_ibge.drop(columns=['Município', 'Distrito', 'Código de Distrito Completo','Nome_Distrito', 'OBS', 'Código de Distrito Completo.1', 'Nome_Distrito.1'])

6.2.Mostrando como ficou a base de pois da exclusão das colunas

In [60]:
df_ibge.head()

,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Código Município Completo,Nome_Município
0,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste
1,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste
2,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste
3,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste
4,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,1100015,Alta Floresta D'Oeste


7.Salvando nova base de dados do IBGE

In [61]:
df_ibge.to_csv('dados_tratados_ibge.csv', index=False)



7.1.Carregando as novas bases tratadas

In [62]:
sus = pd.read_csv("dados_sus_tratados.csv", dtype=str)
ibge = pd.read_csv("dados_tratados_ibge.csv", dtype=str)

7.2.Esse código padroniza os códigos de município para 7 dígitos, preenchendo com zero à esquerda

In [63]:
sus['ID_MUNICIP'] = sus['ID_MUNICIP'].str.zfill(7)
ibge['Código Município Completo'] = ibge['Código Município Completo'].str.zfill(7)


In [64]:
ibge.columns

Index(['UF', 'Nome_UF', 'Região Geográfica Intermediária',
       'Nome Região Geográfica Intermediária', 'Região Geográfica Imediata',
       'Nome Região Geográfica Imediata', 'Código Município Completo',
       'Nome_Município'],
      dtype='object')

In [65]:
sus.head()

,ID_MUNICIP,ID_UNIDADE,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ANO_NASC,CS_SEXO,SG_UF,DT_OBITO
0,2926301,2601567.0,20000809,2000.0,BA,1972.0,F,BA,NaN
1,2901502,2660229.0,20001122,2000.0,BA,1984.0,M,BA,NaN
2,2910800,2799758.0,20001215,2000.0,BA,1958.0,F,BA,NaN
3,2104552,2464659.0,20000810,2000.0,MA,1998.0,F,MA,NaN
4,2105302,2456540.0,20001016,2000.0,MA,1999.0,F,MA,NaN


In [66]:
cols = ['NU_ANO', 'ID_UNIDADE', 'ANO_NASC', 'DT_OBITO']

sus[cols] = sus[cols].apply(
    lambda c: pd.to_numeric(c, errors='coerce')
)

In [67]:
sus[cols] = sus[cols].astype('Int64')

In [68]:
sus.head()


,ID_MUNICIP,ID_UNIDADE,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ANO_NASC,CS_SEXO,SG_UF,DT_OBITO
0,2926301,2601567,20000809,2000,BA,1972,F,BA,<NA>
1,2901502,2660229,20001122,2000,BA,1984,M,BA,<NA>
2,2910800,2799758,20001215,2000,BA,1958,F,BA,<NA>
3,2104552,2464659,20000810,2000,MA,1998,F,MA,<NA>
4,2105302,2456540,20001016,2000,MA,1999,F,MA,<NA>


In [69]:
sus['DT_NOTIFIC'] = pd.to_datetime(sus['DT_NOTIFIC'], format='%Y%m%d', errors='coerce')

In [70]:
sus.head()

,ID_MUNICIP,ID_UNIDADE,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ANO_NASC,CS_SEXO,SG_UF,DT_OBITO
0,2926301,2601567,2000-08-09,2000,BA,1972,F,BA,<NA>
1,2901502,2660229,2000-11-22,2000,BA,1984,M,BA,<NA>
2,2910800,2799758,2000-12-15,2000,BA,1958,F,BA,<NA>
3,2104552,2464659,2000-08-10,2000,MA,1998,F,MA,<NA>
4,2105302,2456540,2000-10-16,2000,MA,1999,F,MA,<NA>


In [71]:
sus['DT_NOTIFIC'] = sus['DT_NOTIFIC'].dt.strftime('%d/%m/%Y')


In [72]:
sus.head()

,ID_MUNICIP,ID_UNIDADE,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ANO_NASC,CS_SEXO,SG_UF,DT_OBITO
0,2926301,2601567,09/08/2000,2000,BA,1972,F,BA,<NA>
1,2901502,2660229,22/11/2000,2000,BA,1984,M,BA,<NA>
2,2910800,2799758,15/12/2000,2000,BA,1958,F,BA,<NA>
3,2104552,2464659,10/08/2000,2000,MA,1998,F,MA,<NA>
4,2105302,2456540,16/10/2000,2000,MA,1999,F,MA,<NA>


In [73]:
sus_merge = sus.merge(
    ibge[['Código Município Completo', 'Nome_Município', 'UF']],
    left_on='ID_MUNICIP',
    right_on='Código Município Completo',
    how='left'
)

In [74]:
sus_merge = sus_merge[
    ['ID_MUNICIP', 'Nome_Município', 'UF',
     'DT_NOTIFIC', 'NU_ANO', 'ANO_NASC', 'CS_SEXO',
     'ID_UNIDADE', 'DT_OBITO']
]


In [75]:
sus_merge.head()

,ID_MUNICIP,Nome_Município,UF,DT_NOTIFIC,NU_ANO,ANO_NASC,CS_SEXO,ID_UNIDADE,DT_OBITO
0,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,<NA>
1,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,<NA>
2,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,<NA>
3,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,<NA>
4,2901502,Anguera,29,22/11/2000,2000,1984,M,2660229,<NA>


In [76]:
sus_merge['DT_OBITO'] = pd.to_datetime(sus_merge['DT_OBITO'], format='%Y%m%d', errors='coerce')

In [77]:
sus_merge.head()

,ID_MUNICIP,Nome_Município,UF,DT_NOTIFIC,NU_ANO,ANO_NASC,CS_SEXO,ID_UNIDADE,DT_OBITO
0,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,NaT
1,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,NaT
2,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,NaT
3,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,NaT
4,2901502,Anguera,29,22/11/2000,2000,1984,M,2660229,NaT


In [78]:
sus_merge['DT_OBITO'] = sus_merge['DT_OBITO'].dt.strftime('%d/%m/%Y')

In [79]:
sus_merge.head()

,ID_MUNICIP,Nome_Município,UF,DT_NOTIFIC,NU_ANO,ANO_NASC,CS_SEXO,ID_UNIDADE,DT_OBITO
0,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,NaN
1,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,NaN
2,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,NaN
3,2926301,Riachão do Jacuípe,29,09/08/2000,2000,1972,F,2601567,NaN
4,2901502,Anguera,29,22/11/2000,2000,1984,M,2660229,NaN


In [80]:
import pdfplumber
import pandas as pd

dados = []

with pdfplumber.open("Módulo Impressão Relatório.pdf") as pdf:
    for page in pdf.pages:
        texto = page.extract_text()
        if not texto:
            continue

        linhas = texto.split("\n")

        for linha in linhas:
            linha = linha.strip()

            # pega linhas que começam com número (CNES)
            if linha and linha[0].isdigit():
                partes = linha.split(" ", 1)

                if len(partes) == 2:
                    cnes, nome = partes
                    dados.append({
                        "CNES": cnes,
                        "ESTABELECIMENTO": nome
                    })

cnes = pd.DataFrame(dados)


In [81]:
cnes.head(10)

,CNES,ESTABELECIMENTO
0,"1/10/26,",5:18 PM Módulo Impressão Relatório
1,3357783,CLINICA ENIO SERRA
2,2696851,HOSPITAL SAO GONCALO LTDA
3,2270234,SES RJ HOSPITAL ESTADUAL GETULIO VARGAS
4,2267802,HOSPITAL GERAL DE ARRAIAL DO CABO
5,2270676,SANTA CASA DA MISERICORDIA DO RIO DE JANEIRO
6,5731186,MATERNIDADE MUNICIPAL DE CARIACICA
7,2550687,HOSPITAL DR ROBERTO ARNIZAUT SILVARES
8,3022609,VIVENCIA PSIQUIATRIA DINAMICA
9,3024660,HOSPITAL ASSUNCAO
